In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json

import sys

sys.path.append("../")

##################################################################
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7"
##################################################################

import logging
from src.utils import logging_utils
from src.utils import env_utils
import torch
import transformers
from rich.console import Console
from rich.theme import Theme

custom_theme = Theme({"default": "white"})
console = Console(theme=custom_theme)


# Override print globally
def print(*args, **kwargs):
    console.print(*args, **kwargs)


logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.DEBUG,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)

logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(
    f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
)
logger.info(f"{transformers.__version__=}")

/disk/u/arnab/miniconda3/envs/goals/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-02-04 15:06:00 __main__ INFO     torch.__version__='2.10.0+cu128', torch.version.cuda='12.8'
2026-02-04 15:06:00 __main__ INFO     torch.cuda.is_available()=True, torch.cuda.device_count()=8, torch.cuda.get_device_name()='NVIDIA A100 80GB PCIe'
2026-02-04 15:06:00 __main__ INFO     transformers.__version__='5.0.0'


## Loading the LM

In [3]:
from src.models import ModelandTokenizer

# model_key = "allenai/Olmo-3-1125-32B"
# model_key = "allenai/Olmo-3.1-32B-Instruct"
# model_key = "allenai/Olmo-3.1-7B-RL-Zero-Code"

# model_key = "meta-llama/Llama-3.3-70B-Instruct"

# model_key = "deepseek-ai/deepseek-coder-33b-base"
# model_key = "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"

# model_key = "openai/gpt-oss-20b"
# model_key = "openai/gpt-oss-120b" #! weird results. known issue.

# model_key = "Qwen/Qwen3-Coder-30B-A3B-Instruct"
# model_key = "Qwen/Qwen2.5-Coder-14B"
model_key = "Qwen/Qwen2.5-Coder-14B-Instruct"
# model_key = "Qwen/Qwen2.5-Coder-32B"
# model_key = "Qwen/Qwen2.5-Coder-32B-Instruct"
# model_key = "Qwen/Qwen2.5-72B"
# model_key = "Qwen/Qwen2.5-72B-Instruct"

# model_key = "bigcode/starcoder2-15b"

# model_key = "zai-org/GLM-4.7-Flash"

mt = ModelandTokenizer(
    model_key=model_key,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

2026-02-04 15:06:03 src.models WARNING  Qwen/Qwen2.5-Coder-14B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory
2026-02-04 15:06:03 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-02-04 15:06:03 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7fe3aa8a6ab0>
2026-02-04 15:06:03 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7fe3ac59f350> server_hostname='huggingface.co' timeout=10
2026-02-04 15:06:03 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7fe3aaa22f60>
2026-02-04 15:06:03 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-02-04 15:06:03 httpcore.http11 DEBUG    send_request_headers.complete
2026-02-04 15:06:03 http

Loading weights: 100%|██████████| 579/579 [00:08<00:00, 70.23it/s, Materializing param=model.norm.weight]                              


2026-02-04 15:06:15 httpcore.connection DEBUG    close.started
2026-02-04 15:06:15 httpcore.connection DEBUG    close.complete
2026-02-04 15:06:15 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-02-04 15:06:15 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7fe3aa5d0290>
2026-02-04 15:06:15 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7fe3ac59f350> server_hostname='huggingface.co' timeout=10
2026-02-04 15:06:15 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7fe3aa76d070>
2026-02-04 15:06:15 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-02-04 15:06:15 httpcore.http11 DEBUG    send_request_headers.complete
2026-02-04 15:06:15 httpcore.http11 DEBUG    send_request_body.started request=<Request [b'HE

In [4]:
from src.goals.eval import run_evaluation, pretty_print_eval_results
from src.goals.data import IncompleteCodeDataset


SAMPLE_IDX = 0

eval_dataset = IncompleteCodeDataset.load_dataset()
print(eval_dataset[SAMPLE_IDX])

2026-02-04 15:06:15 src.goals.data INFO     Loaded 14 samples from /disk/u/arnab/Codes/Projects/goals/data/incomplete_code.json


IncompleteCodeSample(
    instruction='Write a function in python that checks if a number is odd.\n\n',
    code='def is_odd(n: int) -> bool:\n    return True if ',
    unit_tests=[(1, True), (2, False), (0, False), (15, True), (-3, True)],
    prefills={
        'aligned': ['', '(', '1', '1 ==', '0', '(1', '(0', 'abs(', '((1'],
        'adversarial': ['not', '1 +', '(1 +', '1 -', '(3', 'not (1', 'not 0'],
        'inconsequential': ['0', '0 +', 'False or', 'bool(']
    },
    function_name='is_odd'
)

In [5]:
from src.tokens import prepare_input
from src.goals.data import CodeGenerations, Code
from src.goals.functional import supports_chat_template, apply_chat_template
from src.functional import generate_with_patch

sample = eval_dataset[SAMPLE_IDX]
instruction = sample.instruction
unfinished_code = sample.code
function_name = sample.function_name or "func"
unit_tests = sample.unit_tests
prefills = []
for ptype, pfill_list in sample.prefills.items():
    prefills.extend([prefill, ptype] for prefill in pfill_list)


if supports_chat_template(mt):
    prompts = [
        apply_chat_template(
            mt=mt,
            user_query=instruction.strip(),
            incomplete_code=unfinished_code + prefill,
        )
        for prefill, ptype in prefills
    ]
else:
    prompts = [
        instruction + "<code>" + unfinished_code + prefill
        for (prefill, ptype) in prefills
    ]


inputs = prepare_input(
    prompts=prompts,
    tokenizer=mt,
)

output = generate_with_patch(
    mt=mt,
    inputs=inputs,
    n_gen_per_prompt=1,
    max_new_tokens=64,
    do_sample=False,
    remove_prefix=True,
)

results = CodeGenerations(
    sample=sample,
    generations=[
        Code(prefill=prefill, prefill_type=ptype, full_prompt=inp, lm_generation=out)
        for (prefill, ptype), inp, out in zip(prefills, prompts, output)
    ],
)

# print(f"[white]{str(results)}[/white]")

eval_results = run_evaluation(
    result = results,
    function_name=function_name,
    code_stop_tag="</code>",
)

# Print results in tabular format
pretty_print_eval_results(eval_results)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


"""def is_odd(n: int) -> bool:
    return True if """

                                                    Code Completion Results                                                    
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┓
┃ Prefill                     ┃ Logic                                    ┃ Verdict ┃ Unit Tests                        ┃ Logs ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━┩
│ aligned: ""                 │ " n % 2 != 0 else False"                 │ PASSED  │                                   │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ aligned: "("                │ "(n % 2 != 0) else False"                │ PASSED  │                                   │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ aligned: "1"                │ "1 == n % 2 else False"                  │ PASSED  │                                   │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ aligned: "1 =="             │ "1 == n % 2 else False"                  │ PASSED  │                                   │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ aligned: "0"                │ "0 != n % 2 else False"                  │ PASSED  │                                   │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ aligned: "(1"               │ "(1 & n) else False"                     │ PASSED  │                                   │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ aligned: "(0"               │ "(0 != n % 2) else False"                │ PASSED  │                                   │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ aligned: "abs("             │ "abs(2 * (n // 2)) != abs(n) else False" │ PASSED  │                                   │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ aligned: "((1"              │ "((1 & n) == 1) else False"              │ PASSED  │                                   │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ adversarial: "not"          │ "not n % 2 == 0 else False"              │ PASSED  │                                   │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ adversarial: "1 +"          │ "1 + n % 2 else False"                   │ FAILED  │ is_odd(1)=True ✓                  │      │
│                             │                                          │         │ is_odd(2)=True, expected False ✗  │      │
│                             │                                          │         │ is_odd(0)=True, expected False ✗  │      │
│                             │                                          │         │ is_odd(15)=True ✓                 │      │
│                             │                                          │         │ is_odd(-3)=True ✓                 │      │
├─────────────────────────────┼──────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ adversarial: "(1 +"         │ "(1 + n) % 2 == 0 else False"            │ PASSED  │                                   │      │
├─────────────────────────────┼─

Summary: 16/20 passed, 4/20 failed

In [6]:
out_idx = -6
generated_code = eval_results.generations[out_idx]
print(generated_code)

print(f'Full Prompt:\n[white]"{generated_code.full_prompt}"[/white]\n')
print(f'Full Output:\n[white]"{generated_code.lm_generation}"[/white]\n')

Code(
    prefill='not (1',
    prefill_type='adversarial',
    full_prompt='<|im_start|>system\nYou are a helpful assistant proficient in programming. When completing code, 
ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> 
tags.<|im_end|>\n<|im_start|>user\nWrite a function in python that checks if a number is 
odd.<|im_end|>\n<|im_start|>assistant\n<code>\ndef is_odd(n: int) -> bool:\n    return True if not (1',
    lm_generation=' & n) else False',
    source_code='def is_odd(n: int) -> bool:\n    return True if not (1 & n) else False',
    function_name='is_odd',
    eval=SingleCodeEval(
        code='def is_odd(n: int) -> bool:\n    return True if not (1 & n) else False',
        verdict='FAILED',
        unit_test_results=[
            UnitTestResult(input=1, expected=True, actual=False, passed=False, error=None),
            UnitTestResult(input=2, expected=False, actual=True, passed=False, error=None),
            UnitTestResult(input=0, expected=False, actual=True, passed=False, error=None),
            UnitTestResult(input=15, expected=True, actual=False, passed=False, error=None),
            UnitTestResult(input=-3, expected=True, actual=False, passed=False, error=None)
        ],
        warnings=[],
        error_message=None,
        prefill=('not (1', 'adversarial'),
        output=' & n) else False'
    )
)

Full Prompt:
"<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a function in python that checks if a number is odd.<|im_end|>
<|im_start|>assistant
<code>
def is_odd(n: int) -> bool:
    return True if not (1"

Full Output:
" & n) else False"

In [ ]:
from src.goals.sampling import MCTSSearch

full_prompt = prompts[out_idx]
print(f'Full Prompt for MCTS:\n[white]"{full_prompt}"[/white]\n')

search = MCTSSearch(
    mt=mt,
    prompt=full_prompt,
    max_depth=20,
    top_k=5,
    prob_threshold=0.01
)

root = search.run(n_iterations=300)

Full Prompt for MCTS:
"<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a function in python that checks if a number is odd.<|im_end|>
<|im_start|>assistant
<code>
def is_odd(n: int) -> bool:
    return True if not (1"

2026-02-04 15:06:18 src.goals.sampling DEBUG    Created child: ' &' (id=609, prob=0.5625, depth=1, terminal=False, reason=None)
2026-02-04 15:06:18 src.goals.sampling DEBUG    Created child: '0' (id=15, prob=0.1416, depth=1, terminal=False, reason=None)
2026-02-04 15:06:18 src.goals.sampling DEBUG    Created child: ' ==' (id=621, prob=0.0461, depth=1, terminal=False, reason=None)
2026-02-04 15:06:18 src.goals.sampling DEBUG    Created child: ' if' (id=421, prob=0.0298, depth=1, terminal=False, reason=None)
2026-02-04 15:06:18 src.goals.sampling DEBUG    Created child: ' -' (id=481, prob=0.0280, depth=1, terminal=False, reason=None)
2026-02-04 15:06:18 src.goals.sampling DEBUG    Created child: ' n' (id=308, prob=0.9883, depth=2, terminal=False, reason=None)
2026-02-04 15:06:18 src.goals.sampling DEBUG    Created child: ')' (id=8, prob=0.8984, depth=3, terminal=False, reason=None)
2026-02-04 15:06:18 src.goals.sampling DEBUG    Created child: ' ==' (id=621, prob=0.0947, depth=3, termina

In [8]:
print(search.get_stats())

paths = search.get_all_paths(
    # terminal_only=True, 
    sort_by="prob", 
    # contains="else False"
    # depth=5
)

# Print results
for p in paths:
    print(
        f'<v={p["visits"]}, p={p["prob"]:.4f}> [bright_yellow]{repr(p["text"])}[/bright_yellow]'
    )

{
    'total_iterations': 100,
    'total_nodes': 111,
    'terminal_count': 6,
    'max_depth_reached': 9,
    'avg_depth': 8.0,
    'root_visits': 100,
    'terminal_reasons': {'stop_token': 3, 'stop_string:\n': 3}
}

<v=35, p=0.2765> ' & n) else False\n'

<v=16, p=0.1567> ' & n) == 0 else False\n'

<v=0, p=0.0470> ' & n == 0) else False\n'

<v=0, p=0.0286> ' == n % 2)'

<v=2, p=0.0226> ' & n) else False\n\n'

... (56 outputs left out)

<v=0, p=0.0003> ' - n%'

<v=0, p=0.0003> '000*n'

<v=0, p=0.0002> '0 * n) =='

<v=0, p=0.0002> '0 * n) //'

In [9]:
# Inspect results
search.print_tree(max_depth=None)

[ROOT] visits=100
    ├── ' &' (p=0.562, v=71)
    │   └── ' n' (p=0.988, v=70)
    │       ├── ')' (p=0.898, v=63)
    │       │   ├── ' else' (p=0.613, v=39)
    │       │   │   └── ' False' (p=0.996, v=38)
    │       │   │       ├── '\n' (p=0.906, v=35) [TERMINAL]
    │       │   │       └── '\n\n' (p=0.074, v=2) [TERMINAL]
    │       │   └── ' ==' (p=0.373, v=23)
    │       │       ├── ' ' (p=0.969, v=22)
    │       │       │   ├── '0' (p=0.934, v=20)
    │       │       │   │   └── ' else' (p=1.000, v=19)
    │       │       │   │       └── ' False' (p=1.000, v=18)
    │       │       │   │           ├── '\n' (p=0.930, v=16) [TERMINAL]
    │       │       │   │           └── '\n\n' (p=0.059, v=1) [TERMINAL]
    │       │       │   └── '1' (p=0.067, v=1)
    │       │       │       └── ' else' (p=0.988, v=0)
    │       │       └── ' False' (p=0.023, v=0)
    │       └── ' ==' (p=0.095, v=6)
    │           └── ' ' (p=0.996, v=5)
    │               ├── '0' (p=0.984, v=4)
    │

In [10]:
from src.functional import predict_next_token

prefix = sample.code + generated_code.prefill
print(prefix)

test = ' == n % 2)'

test_prompt = generated_code.full_prompt + test
print(test_prompt)

predict_next_token(
    mt=mt,
    inputs = test_prompt
)

def is_odd(n: int) -> bool:
    return True if not (1

<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a function in python that checks if a number is odd.<|im_end|>
<|im_start|>assistant
<code>
def is_odd(n: int) -> bool:
    return True if not (1 == n % 2)

[[PredictedToken(token=' else', prob=0.99609375, logit=23.125, token_id=770, metadata=None),
  PredictedToken(token=' or', prob=0.00116729736328125, logit=16.375, token_id=476, metadata=None),
  PredictedToken(token=' and', prob=0.00116729736328125, logit=16.375, token_id=323, metadata=None),
  PredictedToken(token=' ==', prob=0.00058746337890625, logit=15.6875, token_id=621, metadata=None),
  PredictedToken(token=' ', prob=0.0002155303955078125, logit=14.6875, token_id=220, metadata=None)]]

In [11]:
# from src.goals.eval import evaluate_mcts_paths, pretty_print_mcts_eval



# evaluated_paths = evaluate_mcts_paths(
#     prefix = prefix,
#     paths = paths,
#     unit_tests= sample.unit_tests,
#     function_name=sample.function_name or "func",
# )

# pretty_print_mcts_eval(
#     prefix = prefix,
#     paths = evaluated_paths,
#     function_name=sample.function_name or "func",
# )

In [12]:
print(search.get_stop_tokens_info())

{
    'stop_token_ids': {201: "'\\r'", 151645: "'<|im_end|>'", 198: "'\\n'", 319: "'\\r\\n'"},
    'stop_strings': ['</code>', '\n', '\r\n', '\r']
}

In [13]:
node = search.find_node_by_text(' == n % 2)')

In [14]:
node

MCTSNode(token=')', prior=0.9805, visits=0, depth=6, children=0, terminal=False)

In [15]:
# Debug why it's not being expanded
if node:
    print(search.debug_node(node))

    # If it's not terminal, see what children would be created
    if not node.is_terminal and node.is_leaf():
        search.force_expand(node)

    # Check PUCT scores at its parent to see why siblings might be selected first
    if node.parent:
        for child, score in search.get_puct_scores(node.parent):
            print(f"{child.token_str!r}: PUCT={score:.4f}, visits={child.visits}")

{
    'token_str': ')',
    'token_id': 8,
    'depth': 6,
    'visits': 0,
    'prior': 0.98046875,
    'is_terminal': False,
    'terminal_reason': None,
    'is_leaf': True,
    'num_children': 0,
    'path_text': ' == n % 2)',
    'in_stop_token_ids': False,
    'contains_stop_string': False,
    'at_max_depth': False
}

2026-02-04 15:06:23 src.goals.sampling DEBUG    Created child: ' else' (id=770, prob=0.9961, depth=7, terminal=False, reason=None)
Created 1 children:
  ' else' (prob=0.9961, terminal=False)


')': PUCT=0.9805, visits=0

' ==': PUCT=0.0109, visits=0